In [1]:
import os
from pathlib import Path
# 创建项目目录结构
# 获取当前脚本所在目录的绝对路径
current_dir = Path.cwd()  # 或者 Path(".").absolute()
project_root = current_dir / "ship_detection"
dataset_dir = project_root / "datasets" / "SSDD"
models_dir = project_root / "models"
results_dir = project_root / "results"

print(f"项目根目录: {project_root}")

项目根目录: D:\ShipTarget\ship_detection


In [2]:
from ultralytics import YOLO
import yaml

# 加载数据集配置
with open(dataset_dir / "data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)

print("数据集配置:")
print(f"类别数量: {data_config['nc']}")
print(f"类别名称: {data_config['names']}")
print(f"训练图像路径: {dataset_dir / data_config['train']}")
print(f"验证图像路径: {dataset_dir / data_config['val']}")

# 验证目录是否存在
train_img_dir = dataset_dir / data_config['train']
assert train_img_dir.exists(), f"训练图像目录不存在: {train_img_dir}"

数据集配置:
类别数量: 1
类别名称: ['ship']
训练图像路径: D:\ShipTarget\ship_detection\datasets\SSDD\images\train
验证图像路径: D:\ShipTarget\ship_detection\datasets\SSDD\images\val


In [3]:
# 加载预训练模型
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 设置 HF 镜像
model = YOLO('yolo11s.pt')  # 将自动下载预训练权重
print("模型加载成功")

模型加载成功


In [4]:
# 开始训练
results = model.train(
    data=str(dataset_dir / "data.yaml"),  # 数据集配置
    epochs=100,                            # 训练轮数（根据开题报告）
    imgsz=320,                              # 输入图像尺寸
    batch=8,                                # 批次大小（根据显存调整）
    workers=2,                                # 数据加载线程数
    device=0,                                 # cpu
    project=str(results_dir / "yolov11"),    # 结果保存目录
    name="baseline_SSDD",                          # 实验名称
    exist_ok=True,                             # 允许覆盖
    patience=20,                                # 早停耐心值
    save=True,                                   # 保存模型
    save_period=10,                               # 每10轮保存一次
    plots=True,                                    # 生成训练图表
    cache=False,                                     # 缓存数据加速关闭，减轻压力
)

print("YOLOv11训练完成")

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ShipTarget\ship_detection\datasets\SSDD\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic

In [11]:
# 在测试集上评估模型
best_model_path = results_dir / "yolov11" / "baseline_SSDD" / "weights" / "best.pt"
model = YOLO(best_model_path)

# 评估
metrics = model.val(
    data=str(dataset_dir / "data.yaml"),
    split='val',  # 使用测试集
    batch=2,
    imgsz=320,
    conf=0.25,     # 置信度阈值
    iou=0.5,       # IoU阈值
)

print("YOLOv11评估结果:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"召回率: {metrics.box.mr:.4f}")
print(f"精确率: {metrics.box.mp:.4f}")

Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 559.7138.7 MB/s, size: 42.7 KB)
val: Scanning D:\ShipTarget\ship_detection\datasets\SSDD\labels\val.cache... 232 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 232/232  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 49.3it/s 2.4s0.1s
                   all        232        546      0.959      0.914      0.963      0.717
Speed: 0.3ms preprocess, 5.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to D:\ShipTarget\runs\detect\val24
YOLOv11评估结果:
mAP50: 0.9628
mAP50-95: 0.7173
召回率: 0.9139
精确率: 0.9592


In [17]:
def train_improved_model():
    custom_yaml = project_root / "custom_yolov11.yaml"
    model = YOLO(str(custom_yaml)).load("yolo11s.pt")
    
    # 开始训练
    results = model.train(
        data=str(dataset_dir / "data.yaml"),
        epochs=200,                # 适当增加轮数
        imgsz=320,
        batch=1,
        workers=1,
        device=0,                  
        project=str(results_dir / "improved"),
        name="mmsa_SSDD",
        exist_ok=True,
        patience=30,
        save=True,
        save_period=10,
        plots=True,
        cache=False,
        amp=True,
        optimizer='MuSGD',
        lr0=0.01,                  # 初始学习率
        lrf=0.01,                  # 最终学习率
        momentum=0.9,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        box=7.5,                   # 边界框损失权重
        cls=0.5,                   # 分类损失权重
        dfl=1.5,                   # DFL损失权重
    )
    
    print("改进模型训练完成")
    return

In [18]:
train_improved_model()

Transferred 54/381 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ShipTarget\ship_detection\datasets\SSDD\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/200      5.69G      2.298      1.516      1.778          3        320: 100% ━━━━━━━━━━━━ 928/928 7.8it/s 1:58<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 24.7it/s 4.7s0.1s
                   all        232        546    0.00573      0.108    0.00218   0.000555

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/200      1.29G      4.704      2.911      2.629          4        320: 0% ──────────── 1/928 2.1it/s 0.3s<7:20

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/200      1.29G      3.001      1.852       1.95         10        320: 100% ━━━━━━━━━━━━ 928/928 13.8it/s 1:07<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 29.2it/s 4.0s0.1s
                   all        232        546     0.0112      0.326    0.00775     0.0024

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/200      1.29G      2.748       2.93      1.674          2        320: 0% ──────────── 1/928 2.3it/s 0.1s<6:45

...

    198/200      1.66G     0.8782     0.4235     0.8668          1        320: 100% ━━━━━━━━━━━━ 928/928 14.5it/s 1:04<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 30.8it/s 3.8s0.1s
                   all        232        546      0.953      0.899      0.954      0.679

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    199/200      1.66G     0.8702     0.4182     0.8608          3        320: 100% ━━━━━━━━━━━━ 928/928 14.7it/s 1:03<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 30.6it/s 3.8s0.0s
                   all        232        546      0.956      0.903      0.955      0.677

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    200/200      1.66G     0.9112     0.4731     0.8501          2        320: 0% ──────────── 3/928 6.5it/s 0.3s<2:23

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    200/200      1.66G     0.8888     0.4245     0.8699          1        320: 100% ━━━━━━━━━━━━ 928/928 14.5it/s 1:04<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 29.7it/s 3.9s0.0s
                   all        232        546      0.945      0.897      0.955      0.676

200 epochs completed in 3.819 hours.
Optimizer stripped from D:\ShipTarget\ship_detection\results\improved\mmsa_SSDD\weights\last.pt, 23.7MB
Optimizer stripped from D:\ShipTarget\ship_detection\results\improved\mmsa_SSDD\weights\best.pt, 23.7MB

Validating D:\ShipTarget\ship_detection\results\improved\mmsa_SSDD\weights\best.pt...
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
custom_YOLOv11 summary (fused): 101 layers, 11,711,001 parameters, 0 gradients, 31.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 52.4i

In [12]:
# 在测试集上评估模型
best_model_path = results_dir / "improved" / "mmsa_SSDD" / "weights" / "best.pt"
model = YOLO(best_model_path)

# 评估
metrics = model.val(
    data=str(dataset_dir / "data.yaml"),
    split='val',  # 使用测试集
    batch=2,
    imgsz=320,
    conf=0.25,     # 置信度阈值
    iou=0.5,       # IoU阈值
)

print("YOLOv11评估结果:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"召回率: {metrics.box.mr:.4f}")
print(f"精确率: {metrics.box.mp:.4f}")

Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
custom_YOLOv11 summary (fused): 101 layers, 11,711,001 parameters, 0 gradients, 31.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 484.7232.3 MB/s, size: 37.6 KB)
val: Scanning D:\ShipTarget\ship_detection\datasets\SSDD\labels\val.cache... 232 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 232/232  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 45.3it/s 2.6s0.1s
                   all        232        546       0.98      0.885      0.939      0.706
Speed: 0.3ms preprocess, 7.0ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to D:\ShipTarget\runs\detect\val25
YOLOv11评估结果:
mAP50: 0.9387
mAP50-95: 0.7059
召回率: 0.8846
精确率: 0.9797


In [24]:
def train_improved_model_2():
    custom_yaml = project_root / "custom_yolov11.yaml"
    model = YOLO(str(custom_yaml)).load("yolo11s.pt")
    
    # 开始训练
    results = model.train(
        data=str(dataset_dir / "data.yaml"),
        epochs=200,                # 适当增加轮数
        imgsz=320,
        batch=1,
        workers=1,
        device=0,                  
        project=str(results_dir / "improved"),
        name="mmsa_SSDD_2",
        exist_ok=True,
        patience=30,
        save=True,
        save_period=10,
        plots=True,
        cache=False,
        amp=True,
        optimizer='AdamW',
        lr0=0.001,                  # 初始学习率
        lrf=0.01,                  # 最终学习率
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        box=7.5,                   # 边界框损失权重
        cls=0.5,                   # 分类损失权重
        dfl=1.5,                   # DFL损失权重
    )
    
    print("改进模型训练完成")
    return

In [25]:
train_improved_model_2()

Transferred 54/381 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ShipTarget\ship_detection\datasets\SSDD\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/200       8.9G      3.508       2.07      2.036          3        320: 100% ━━━━━━━━━━━━ 928/928 9.0it/s 1:43<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 23.4it/s 5.0s0.0s
                   all        232        546      0.261      0.375      0.172     0.0529

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/200      8.62G      2.435      1.764      1.884          3        320: 0% ──────────── 2/928 3.0it/s 0.3s<5:11

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/200      8.62G      2.736      1.566      1.688         10        320: 100% ━━━━━━━━━━━━ 928/928 11.9it/s 1:18<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 27.6it/s 4.2s0.0s
                   all        232        546      0.469      0.458      0.382      0.111

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/200      8.62G      2.914       2.43      1.707          5        320: 0% ──────────── 2/928 3.8it/s 0.3s<4:05

...

    116/200      8.62G      1.175     0.6184     0.9345          2        320: 100% ━━━━━━━━━━━━ 928/928 12.1it/s 1:17<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 27.4it/s 4.2s0.1s
                   all        232        546      0.889      0.876      0.914      0.592

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    117/200      8.62G      1.133     0.5689     0.8835          3        320: 0% ──────────── 1/928 1.5it/s 0.2s<10:13

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    117/200      8.62G      1.166     0.6238     0.9423          3        320: 100% ━━━━━━━━━━━━ 928/928 12.1it/s 1:17<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 27.5it/s 4.2s0.0s
                   all        232        546      0.915      0.875      0.922      0.597

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    118/200      8.62G     0.9593      0.498      1.048          6        320: 0% ──────────── 1/928 1.8it/s 0.2s<8:32

D:\anaconda3\envs\yolo11\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    118/200      8.62G      1.146     0.6058     0.9352          2        320: 100% ━━━━━━━━━━━━ 928/928 12.1it/s 1:17<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 27.3it/s 4.2s0.1s
                   all        232        546      0.933      0.873      0.927      0.619
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 88, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

118 epochs completed in 2.688 hours.
Optimizer stripped from D:\ShipTarget\ship_detection\results\improved\mmsa_SSDD_2\weights\last.pt, 23.7MB
Optimizer stripped from D:\ShipTarget\ship_detection\results\improved\mmsa_SSDD_2\weights\best.pt, 23.7MB

Validating D:\ShipTarget\ship_detection\results\improved\mmsa_SSDD_2\weights\best.pt...
Ultralytics 8.4.16  Python-3.11.14 

In [10]:
# 在测试集上评估模型
best_model_path = results_dir / "improved" / "mmsa_SSDD_2" / "weights" / "best.pt"
model = YOLO(best_model_path)

# 评估
metrics = model.val(
    data=str(dataset_dir / "data.yaml"),
    split='val',  # 使用测试集
    batch=2,
    #imgsz=640,
    conf=0.25,     # 置信度阈值
    iou=0.5,       # IoU阈值
)

print("YOLOv11评估结果:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"召回率: {metrics.box.mr:.4f}")
print(f"精确率: {metrics.box.mp:.4f}")

Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
custom_YOLOv11 summary (fused): 101 layers, 11,711,001 parameters, 0 gradients, 31.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 513.4121.8 MB/s, size: 39.5 KB)
val: Scanning D:\ShipTarget\ship_detection\datasets\SSDD\labels\val.cache... 232 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 232/232  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 45.8it/s 2.5s0.0s
                   all        232        546       0.93      0.881      0.928      0.662
Speed: 0.3ms preprocess, 6.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to D:\ShipTarget\runs\detect\val23
YOLOv11评估结果:
mAP50: 0.9283
mAP50-95: 0.6621
召回率: 0.8810
精确率: 0.9304
